In [1]:
!ollama list

NAME               ID              SIZE      MODIFIED    
gpt-oss:latest     17052f91a42e    13 GB     3 weeks ago    
llama3.1:latest    46e0c10c039e    4.9 GB    4 weeks ago    
qwen3.5:latest     6488c96fa5fa    6.6 GB    4 weeks ago    
gemma4:latest      c6eb396dbd59    9.6 GB    4 weeks ago    
llama3.1:8b        46e0c10c039e    4.9 GB    4 weeks ago    
qwen3.5:9b         6488c96fa5fa    6.6 GB    4 weeks ago    
gemma4:e4b         c6eb396dbd59    9.6 GB    4 weeks ago    


In [ ]:
from ollama import chat

response = chat(
    model="gemma4:e4b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "2022년 월드컵 우승팀은 어디야?"},
    ],
)

print(response.message.content)

2022년 카타르 월드컵의 우승팀은 **아르헨티나**입니다! 🏆

결승전에서 프랑스를 꺾고 감격적인 우승을 차지했습니다.


## OpenAI-compatible API란?

**OpenAI-compatible API**는 OpenAI API와 동일하거나 매우 유사한 요청 형식과 응답 구조를 제공하는 API를 말합니다. 따라서 `openai` Python SDK로 작성한 코드의 구조를 크게 바꾸지 않고도, OpenAI가 아닌 서비스나 로컬 모델 서버를 호출할 수 있습니다.

Ollama는 로컬에서 실행 중인 모델에 대해 OpenAI 호환 엔드포인트를 제공합니다. 아래 예제에서는 Ollama 전용 Python 라이브러리 대신 `openai` SDK를 사용하여 로컬의 `gemma4:e4b` 모델과 대화합니다.

### 왜 사용하나요?

- **코드 재사용**: `client.chat.completions.create(...)`처럼 OpenAI SDK 문법으로 작성한 애플리케이션 코드를 여러 제공자에 적용하기 쉽습니다.
- **전환이 쉬움**: 개발 중에는 비용 없이 로컬 Ollama 모델을 사용하고, 배포 환경에서는 OpenAI API로 바꾸는 식의 전환이 수월합니다.
- **도구 생태계 활용**: OpenAI SDK를 기준으로 만들어진 예제, 프레임워크, 라이브러리를 활용할 수 있습니다.

### 이 예제의 연결 설정

```python
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)
```

| 설정 | 의미 |
| --- | --- |
| `base_url` | 요청을 보낼 서버의 주소입니다. 기본 OpenAI 서버 대신 로컬 Ollama 서버의 OpenAI 호환 주소를 지정합니다. `/v1`까지 포함해야 합니다. |
| `api_key` | OpenAI API에서는 실제 API 키가 필요합니다. 로컬 Ollama는 일반적으로 이 값을 인증에 사용하지 않으므로, SDK가 요구하는 임의의 문자열을 넣습니다. |
| `model` | Ollama에 이미 내려받아 둔 모델 이름입니다. `ollama list`로 확인한 이름을 사용합니다. |

### 호출 흐름

```text
Python 코드 → openai SDK → http://localhost:11434/v1 → Ollama → gemma4:e4b
```

`messages`는 대화 이력을 역할별로 전달하는 목록입니다. `system` 메시지는 모델의 동작 원칙을 정하고, `user` 메시지는 사용자의 질문을 전달합니다. 생성된 답변은 `response.choices[0].message.content`에서 꺼냅니다.

### 실행 전 확인 사항

1. Ollama가 설치되어 있고 `gemma4:e4b` 모델이 내려받아져 있어야 합니다.
2. Ollama 서버가 실행 중이어야 합니다. 보통 Ollama 앱을 실행하거나 터미널에서 `ollama serve`를 실행합니다.
3. Python 환경에 OpenAI SDK가 없으면 `pip install openai`를 실행합니다.

> 호환 API는 요청 형식이 비슷하다는 뜻이며, 모든 OpenAI 기능과 옵션이 완전히 동일하게 동작한다는 보장은 아닙니다. 사용하려는 기능(예: 스트리밍, 도구 호출, 구조화된 출력)은 Ollama와 모델이 지원하는지 별도로 확인하세요.

In [2]:
from openai import OpenAI

# Ollama의 OpenAI 호환 API에 연결합니다. (ollama serve 실행 필요)
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # 로컬 Ollama에서는 임의의 문자열이면 됩니다.
)

response = client.chat.completions.create(
  model="gemma4:e4b",
  messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "2022년 월드컵 우승팀은 어디야?"},
  ]
)
print(response.choices[0].message.content)

2022년 FIFA 카타르 월드컵 우승팀은 **아르헨티나**입니다. 🏆

(최종 결승전에서 프랑스를 3-3 무승부로 만든 후, 승부차기 끝에 승리했습니다.)
